# 02 — Seleção de Variáveis

## Objetivo

Investigar quais variáveis podem contribuir para a classificação dos registros de desastres nas classes FALTA e EXCESSO.

A seleção considerará a relevância das informações, a qualidade dos dados, a redundância e o risco de vazamento da variável-alvo. As decisões serão documentadas e validadas antes do treinamento dos modelos.


### 1. Carregamento dos dados

Carregamos a planilha corrigida e definimos as classes de interesse, mantendo apenas os registros pertencentes ao escopo da classificação binária.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

ARQUIVO_BASE = (
    ROOT
    / "data"
    / "raw"
    / "BD_Atlas_1991_2025_v1.1_2026.08.06_Consolidado.xlsx"
)

df = pd.read_excel(
    ARQUIVO_BASE,
    sheet_name="Atlas Valores Corrigidos"
)

mapa_classes = {
    "Estiagem e Seca": "FALTA",
    "Enxurradas": "EXCESSO",
    "Inundações": "EXCESSO",
    "Alagamentos": "EXCESSO",
}

df["alvo"] = df["descricao_tipologia"].map(mapa_classes)

df_modelagem = df[df["alvo"].notna()].copy()

print("Registros para análise:", len(df_modelagem))
print("\nDistribuição das classes:")
display(df_modelagem["alvo"].value_counts())

### 2. Identificação de possíveis vazamentos

Identificamos variáveis que revelam diretamente a tipologia do desastre ou podem reproduzir a definição do alvo. Essas variáveis serão investigadas para evitar que o modelo tenha acesso indireto à resposta correta.

In [ ]:
variaveis_suspeitas = [
    "descricao_tipologia",
    "tipologia",
    "Cod_Cobrade",
    "Protocolo_S2iD",
]

display(
    df_modelagem[variaveis_suspeitas]
    .head()
)

### 3. Qualidade das variáveis

Calculamos a quantidade e o percentual de valores ausentes de cada variável para identificar atributos que exigem tratamento ou cuja utilização pode não ser vantajosa.

In [ ]:
resumo_variaveis = pd.DataFrame({
    "tipo": df_modelagem.dtypes.astype(str),
    "valores_ausentes": df_modelagem.isna().sum(),
    "percentual_ausente": (
        df_modelagem.isna().mean() * 100
    ).round(2),
    "valores_distintos": df_modelagem.nunique(dropna=True),
})

resumo_variaveis = resumo_variaveis.sort_values(
    "percentual_ausente",
    ascending=False
)

display(resumo_variaveis)

### 4. Variabilidade das variáveis

Verificamos a quantidade de valores distintos para identificar variáveis constantes ou com pouca variação, que podem ter utilidade limitada para a classificação.


In [ ]:
variaveis_constantes = (
    resumo_variaveis[
        resumo_variaveis["valores_distintos"] <= 1
    ]
)

print("Variáveis constantes ou sem valores observados:")
display(variaveis_constantes)

### 5. Lista inicial de candidatas

Organizamos as variáveis disponíveis, excluindo temporariamente a variável-alvo e os campos suspeitos de revelar diretamente sua classificação. A lista será revisada antes do pré-processamento e do treinamento.


In [ ]:
colunas_excluir = [
    "alvo",
    *variaveis_suspeitas,
]

candidatas = [
    coluna
    for coluna in df_modelagem.columns
    if coluna not in colunas_excluir
]

print("Quantidade inicial de candidatas:", len(candidatas))

display(
    resumo_variaveis.loc[
        resumo_variaveis.index.intersection(candidatas)
    ]
)

### 6. Variáveis candidatas

Nesta etapa, são selecionadas as variáveis candidatas à modelagem, excluindo o alvo e os campos inicialmente identificados como possíveis fontes de vazamento de dados.

In [9]:
# Variáveis com pelo menos 80% de valores ausentes
alta_ausencia = resumo_variaveis[
    resumo_variaveis["percentual_ausente"] >= 80
].copy()

print("Quantidade de variáveis com 80% ou mais de ausência:",
      len(alta_ausencia))

display(alta_ausencia)

Quantidade de variáveis com 80% ou mais de ausência: 5


,tipo,valores_ausentes,percentual_ausente,valores_distintos
DA_Polui/cont do ar,str,51508,99.20,4
DA_Polui/cont do solo,str,51452,99.10,8
DA_Polui/cont da água,str,50802,97.84,8
DA_Incêndi parques/APA's/APP's,str,50467,97.20,6
DA_Dimi/exauri hídrico,str,47960,92.37,8


### 7. Variáveis constantes

Esta etapa identifica variáveis com um único valor observado ou sem valores válidos, que não contribuem para diferenciar as classes e podem ser removidas.

In [10]:
variaveis_constantes = resumo_variaveis[
    resumo_variaveis["valores_distintos"] <= 1
]

print("Variáveis constantes ou sem valores observados:")
display(variaveis_constantes)

Variáveis constantes ou sem valores observados:


,tipo,valores_ausentes,percentual_ausente,valores_distintos


### 8. Listagem das variáveis candidatas

São exibidos os nomes das variáveis candidatas para facilitar a inspeção dos atributos e apoiar a decisão sobre quais serão utilizados na modelagem.

In [11]:
print(f"Total de variáveis candidatas: {len(candidatas)}")

for i, coluna in enumerate(candidatas, start=1):
    print(f"{i:02d}. {coluna}")

Total de variáveis candidatas: 66
01. Nome_Municipio
02. Sigla_UF
03. regiao
04. Data_Registro
05. Data_Evento
06. grupo_de_desastre
07. Cod_IBGE_Mun
08. Setores Censitários
09. Status
10. DH_Descricao
11. DH_MORTOS
12. DH_FERIDOS
13. DH_ENFERMOS
14. DH_DESABRIGADOS
15. DH_DESALOJADOS
16. DH_DESAPARECIDOS
17. DH_AFETADOS_SECA_ESTIAGEM
18. DH_total_danos_humanos_diretos
19. DH_OUTROS AFETADOS
20. DM_Descricao
21. DM_Uni Habita Danificadas
22. DM_Uni Habita Destruidas
23. DM_Uni Habita Valor
24. DM_Inst Saúde Danificadas
25. DM_Inst Saúde Destruidas
26. DM_Inst Saúde Valor
27. DM_Inst Ensino Danificadas
28. DM_Inst Ensino Destruidas
29. DM_Inst Ensino Valor
30. DM_Inst Serviços Danificadas
31. DM_Inst Serviços Destruidas
32. DM_Inst Serviços Valor
33. DM_Inst Comuni Danificadas
34. DM_Inst Comuni Destruidas
35. DM_Inst Comuni Valor
36. DM_Obras de Infra Danificadas
37. DM_Obras de Infra Destruidas
38. DM_Obras de Infra Valor
39. DM_total_danos_materiais
40. DA_Descricao
41. DA_Polui/cont

### 9. Consolidação da análise

São reunidas as variáveis com alta proporção de valores ausentes, as variáveis constantes e os campos suspeitos de vazamento, apoiando a seleção final dos atributos.

In [12]:
print("=== VARIÁVEIS COM ALTA AUSÊNCIA ===")
display(alta_ausencia)

print("\n=== VARIÁVEIS CONSTANTES ===")
display(variaveis_constantes)

print("\n=== VARIÁVEIS SUSPEITAS DE VAZAMENTO ===")
for coluna in variaveis_suspeitas:
    if coluna in df_modelagem.columns:
        print(f"- {coluna}")

=== VARIÁVEIS COM ALTA AUSÊNCIA ===


,tipo,valores_ausentes,percentual_ausente,valores_distintos
DA_Polui/cont do ar,str,51508,99.20,4
DA_Polui/cont do solo,str,51452,99.10,8
DA_Polui/cont da água,str,50802,97.84,8
DA_Incêndi parques/APA's/APP's,str,50467,97.20,6
DA_Dimi/exauri hídrico,str,47960,92.37,8



=== VARIÁVEIS CONSTANTES ===


,tipo,valores_ausentes,percentual_ausente,valores_distintos



=== VARIÁVEIS SUSPEITAS DE VAZAMENTO ===
- descricao_tipologia
- tipologia
- Cod_Cobrade
- Protocolo_S2iD
